# Chapter 5 — Gaia DR4 epoch astrometry: fitting the orbit of Gaia BH3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/gaia-nss-tutorial/blob/main/notebooks/05_dr4_epoch_astrometry.ipynb)

**Goal.** Gaia DR3 published, for each star, the *result* of fitting a model to its measurements. Gaia DR4, scheduled
for 2 December 2026 (https://www.cosmos.esa.int/web/gaia/data-release-4), will also publish the measurements
themselves: the **epoch astrometry**, one along-scan position per CCD crossing, over 66 months. In June 2026 ESA
pre-released these data for 12 sources (https://www.cosmos.esa.int/web/gaia/dr4-prerelease), among them Gaia BH3,
a 33 $M_\odot$ black hole orbiting a metal-poor giant (Gaia Collaboration, Panuzzo et al. 2024,
https://ui.adsabs.harvard.edu/abs/2024A&A...686L...2G). We fit these data ourselves, with the design matrix of
Chapter 1 plus a Keplerian orbit.

**You will learn**
1. what a DR4 epoch-astrometry record contains, and how to turn it into times, scan angles and along-scan positions;
2. to refit the five-parameter single-star solution, and to read its UWE for quasars, single stars and binaries;
3. to fit a photocentre orbit: linear in the Thiele–Innes constants for fixed $(P, e, T_p)$, non-linear in those three;
4. to convert Thiele–Innes constants to Campbell elements, and the orbit to an astrometric mass function and a mass;
5. why DR3, with half the time span, could not find this orbit.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, brentq     # orbit refinement; root of the mass-function equation

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 5.1 What is in an epoch-astrometry record

The file `dr4_prerelease_epoch_astrometry` is the pre-release table with one row per CCD observation (the original
has one row per transit, with arrays over the CCDs). The columns we use:

- `obs_time_tcb` [ns]: observation time, Barycentric Coordinate Time, counted from JD 2455197.5 (2010 January 1,
  0 h); `obs_time_bary_corr` [ns]: the correction to the barycentre of the solar system, as seen from this source;
- `scan_pos_angle` $\theta$ [deg]: position angle of the scan direction, from north through east;
- `centroid_pos_al` $w$ [mas]: the measured along-scan position, relative to the reference point (`ra0`, `dec0`), and
  `centroid_pos_error_al` $\sigma$ [mas] its uncertainty;
- `parallax_factor_al` $f_{\rm AL}$: the along-scan displacement produced by a parallax of 1 mas;
- `used_by_agis_al`: whether the observation was used by the DR4 astrometric solution (outliers and problematic CCDs
  are not). We keep only those.

The model of a single star is exactly that of Chapter 1 (and of ESA's `gaia-supdate` package, https://github.com/esa/gaia-supdate):

$$ w = \Delta\alpha_*\sin\theta + \Delta\delta\cos\theta + \varpi f_{\rm AL} + (\mu_{\alpha*}\sin\theta + \mu_\delta\cos\theta)\,t , $$

with $t$ the time in years from the DR4 reference epoch J2017.5 (JD 2457936.875). DR4 fits a sixth parameter for some
sources (a colour term); we ignore it, which is adequate for these bright, well-measured stars.

In [ ]:
ep = load('dr4_prerelease_epoch_astrometry')
ep = ep[ep.used_by_agis_al].copy()
JD_REF = 2457936.875                             # J2017.5, the DR4 reference epoch
# time in years from J2017.5: origin JD 2455197.5, plus (TCB time + barycentric correction) in ns, 86400e9 ns per day
ep['t'] = (2455197.5 + (ep.obs_time_tcb + ep.obs_time_bary_corr.astype(float)) / 86400e9 - JD_REF) / 365.25
NAMES = {2237987199365376: 'quasar', 60730287810150016: 'quasar', 10973744521070720: 'quasar',
         4181040337841125632: 'HD 183633', 3926186255616949504: 'parallax', 435469040545191680: 'parallax',
         1663617687609809280: 'LSPM J1324+6124', 2309425390592896: 'magnitude', 20694084440761600: 'magnitude',
         3937211745905473024: 'HD 114762', 1457486023639239296: 'Gaia-4', 4318465066420528000: 'Gaia BH3'}
BH3 = 4318465066420528000
bh = ep[ep.source_id == BH3]
print(f'{len(ep):,} CCD observations of {ep.source_id.nunique()} sources; Gaia BH3: {len(bh)} observations in '
      f'{bh.transit_id.nunique()} transits, from {2017.5 + bh.t.min():.2f} to {2017.5 + bh.t.max():.2f}')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(2017.5 + bh.t, bh.centroid_pos_al, '.', ms=3)
ax[0].set(xlabel='time [yr]', ylabel='along-scan position $w$ [mas]', title='Gaia BH3: all CCD observations')
k = bh.transit_id == bh.transit_id.iloc[100]                # one transit
ax[1].errorbar(bh.ccd[k], bh.centroid_pos_al[k], bh.centroid_pos_error_al[k], fmt='o')
ax[1].set(xlabel='CCD number in the transit', ylabel='$w$ [mas]',
          title=f'one transit ({2017.5 + bh.t[k].iloc[0]:.3f}, scan angle {bh.scan_pos_angle[k].iloc[0]:.0f}°)')
plt.tight_layout(); plt.show()

**Figure 5.1.** Left: along-scan positions $w$ of Gaia BH3, all CCD observations used by the DR4 solution, against
time. Right: the CCD observations of a single transit, with their uncertainties.

The positions jump by hundreds of mas between transits (left): $w$ is a projection, and each transit has its own scan
angle, so the large proper motion of this halo star ($\approx 160$ mas/yr) appears with a different sign and size each
time. Within one transit (right) the CCD measurements agree to their uncertainties, ~0.1 mas for this $G = 11.2$ star.

## 5.2 The single-star fit, source by source

We fit the five-parameter model to each of the 12 sources by weighted least squares and compute UWE
$= \sqrt{\chi^2/\nu}$. The pre-release contains three groups chosen by ESA: quasars (no parallax or proper motion
expected), stars selected by parallax or by magnitude, and three known binaries.

In [ ]:
def design5(s):
    """Design matrix (N, 5) of the single-star model, and sin(theta), cos(theta), t for the CCD observations in s."""
    th = np.radians(s.scan_pos_angle.to_numpy(float))
    t = s.t.to_numpy()
    sn, cs = np.sin(th), np.cos(th)
    return np.column_stack([sn, cs, s.parallax_factor_al.to_numpy(float), t * sn, t * cs]), sn, cs, t


def kepler_xy(t, P, e, tp):
    """Elliptical rectangular coordinates X = cos E - e, Y = sqrt(1 - e^2) sin E (t, P, tp in the same units)."""
    M = 2 * np.pi * (t - tp) / P
    E = M.copy()
    for _ in range(60):                          # Newton iterations for E - e sin E = M
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return np.cos(E) - e, np.sqrt(1 - e**2) * np.sin(E)


def fit(s, orbit=None, accel=False):
    """Weighted linear least squares on the CCD observations in s.

    orbit = (P [yr], e, tp [yr from J2017.5]) adds the photocentre orbit, linear in the Thiele-Innes constants
    (A, F, B, G): Delta delta = A X + F Y, Delta alpha* = B X + G Y. accel adds an acceleration (g_alpha*, g_delta)
    [mas/yr^2] as t^2/2 terms. Returns parameters (5 astrometric, then A, F, B, G or the accelerations), chi^2, nu."""
    D, sn, cs, t = design5(s)
    cols = [D]
    if accel:
        cols.append(np.column_stack([t**2 / 2 * sn, t**2 / 2 * cs]))
    if orbit is not None:
        X, Y = kepler_xy(t, *orbit)
        cols.append(np.column_stack([X * cs, Y * cs, X * sn, Y * sn]))     # columns multiplying A, F, B, G
    M = np.hstack(cols)
    w, sig = s.centroid_pos_al.to_numpy(float), s.centroid_pos_error_al.to_numpy(float)
    p, *_ = np.linalg.lstsq(M / sig[:, None], w / sig, rcond=None)         # rows divided by sigma: weights 1/sigma^2
    return p, np.sum(((w - M @ p) / sig)**2), len(w) - M.shape[1]


rows = []
for sid, s in ep.groupby('source_id'):
    p, chi2, nu = fit(s)
    Dw = design5(s)[0] / s.centroid_pos_error_al.to_numpy(float)[:, None]
    err = np.sqrt(np.diag(np.linalg.inv(Dw.T @ Dw)))                       # formal errors: (D^T W D)^-1
    rows.append(dict(source_id=sid, name=NAMES[sid], G=s.g_mag.iloc[0], n_obs=len(s), parallax=p[2],
                     parallax_error=err[2], pmra=p[3], pmdec=p[4], uwe=np.sqrt(chi2 / nu)))
tab = pd.DataFrame(rows).sort_values('uwe')
print(tab.round(3).to_string(index=False))
plt.semilogy(range(len(tab)), tab.uwe, 'o')
plt.xticks(range(len(tab)), tab.name, rotation=45, ha='right'); plt.ylabel('UWE of the 5-parameter fit')
plt.axhline(1.4, color='r', ls='--', lw=1); plt.show()
single = tab[~tab.name.isin(['HD 114762', 'Gaia-4', 'Gaia BH3'])]
qso = tab[tab.name == 'quasar']
# calibration: quasars and stars without known companions are fitted to their uncertainties, and the quasars'
# parallaxes are zero within 3 formal errors
assert single.uwe.between(0.9, 1.15).all() and (np.abs(qso.parallax / qso.parallax_error) < 3).all()

**Figure 5.2.** UWE of the five-parameter fit for the 12 pre-release sources, in increasing order; dashed: UWE = 1.4.

All three quasars and all six stars without known companions have UWE between 0.98 and 1.04: the published
uncertainties describe the scatter of the measurements, and the single-star model is complete for them. The quasars'
parallaxes are zero within their formal errors (column `parallax_error`). The three binaries stand out: Gaia-4, a
$0.64\,M_\odot$ star with an $11.8\,M_{\rm J}$ planet in a 571-d orbit, $a_0 = 0.31$ mas (Stefánsson et al. 2025,
https://ui.adsabs.harvard.edu/abs/2025AJ....169..107S; UWE 1.9), HD 114762 (UWE 10), and Gaia BH3 (UWE 82). For BH3 the
single-star model is wrong by two orders of magnitude in $\chi^2$.

## 5.3 Fitting the orbit

For fixed period $P$, eccentricity $e$ and time of periastron $T_p$, the orbital displacement is linear in the
Thiele–Innes constants $A, B, F, G$ (Chapter 1), so the full model — five astrometric parameters plus four orbital
constants — is again solved by linear least squares. Only $(P, e, T_p)$ need a non-linear search. We scan a grid in the
three, keep the best $\chi^2$, and refine it with the Nelder–Mead simplex method. Keeping the minimum of $\chi^2$ over
$e$ and $T_p$ at each $P$ gives a **periodogram**, $\chi^2_{\min}(P)$.

In [ ]:
def periodogram(s, P_grid, n_e=10, n_tp=24):
    """Minimum chi^2 over a grid in (e, tp) at each period, and the best (chi^2, P, e, tp) overall."""
    best, prof = None, []
    for P in P_grid:
        c = [(fit(s, (P, e, tp))[1], P, e, tp) for e in np.linspace(0.05, 0.9, n_e)
             for tp in np.linspace(0, P, n_tp, endpoint=False)]
        m = min(c)
        prof.append(m[0])
        best = m if best is None or m[0] < best[0] else best
    return np.array(prof), best


def refine(s, start):
    """Nelder-Mead on (P, e, tp), starting from the grid best; chi^2 is minimised over the 9 linear parameters inside."""
    f = lambda q: fit(s, (q[0], np.clip(q[1], 0, 0.99), q[2]))[1]
    return minimize(f, start, method='Nelder-Mead', options=dict(xatol=1e-9, fatol=1e-8, maxiter=6000)).x


P_grid = np.geomspace(1500, 30000, 25) / 365.25              # 1500 to 30000 days, in years
prof_dr4, best = periodogram(bh, P_grid)
q_bh = refine(bh, best[1:])
p_bh, chi2_bh, nu_bh = fit(bh, tuple(q_bh))
print(f'best orbit: P = {q_bh[0] * 365.25:.1f} d, e = {q_bh[1]:.4f}; UWE {np.sqrt(chi2_bh / nu_bh):.2f} '
      f'(single-star fit: {tab.set_index("name").uwe["Gaia BH3"]:.1f})')

The orbit reduces UWE from 82 to 1.7. The remaining excess, about 1.3 in units of the per-observation uncertainty, is
the kind of small calibration and modelling error (we ignore the colour term and use no outlier rejection) that the
DR4 pipeline absorbs as excess noise.

## 5.4 From Thiele–Innes constants to Campbell elements and a mass

The inverse of the Thiele–Innes relations (Halbwachs et al. 2023, Appendix A,
https://ui.adsabs.harvard.edu/abs/2023A&A...674A...9H): with $u = (A^2+B^2+F^2+G^2)/2$ and $v = AG - BF$,

$$ a_0 = \sqrt{u + \sqrt{(u+v)(u-v)}}, \qquad \cos i = v/a_0^2, $$
$$ \omega + \Omega = \operatorname{atan2}(B - F,\, A + G), \qquad \omega - \Omega = \operatorname{atan2}(-B - F,\, A - G). $$

Astrometry alone cannot tell which way the star moves along the line of sight, so $(\omega, \Omega)$ and
$(\omega + 180°, \Omega + 180°)$ describe the same projected orbit; by convention $0 \le \Omega < 180°$. From $a_0$, the
parallax $\varpi$ and $P$ follows the astrometric mass function (Kepler's third law with $a_0/\varpi$ in au and $P$ in
years),

$$ f_M = \left(\frac{a_0}{\varpi}\right)^3 \left(\frac{P}{\rm yr}\right)^{-2} M_\odot = \frac{M_2^3}{(M_1 + M_2)^2}
\quad\text{(dark companion)}, $$

so that with an estimate of the giant's mass $M_1$ we can solve for $M_2$.

In [ ]:
def campbell(A, B, F, G):
    """Campbell elements (a0, i [deg], omega [deg], Omega [deg]) from Thiele-Innes constants, with 0 <= Omega < 180."""
    u, v = (A**2 + B**2 + F**2 + G**2) / 2, A * G - B * F
    a0 = np.sqrt(u + np.sqrt((u + v) * (u - v)))
    i = np.degrees(np.arccos(v / a0**2))
    s, d = np.arctan2(B - F, A + G), np.arctan2(-B - F, A - G)              # omega + Omega, omega - Omega
    om, Om = np.degrees((s + d) / 2), np.degrees((s - d) / 2)
    flip = (Om % 360) >= 180                                                # (omega, Omega) -> (omega + 180, Omega + 180)
    return a0, i, (om + 180 * flip) % 360, (Om + 180 * flip) % 360


A, F, B, G = p_bh[5:9]
a0, inc, om, Om = campbell(A, B, F, G)
plx = p_bh[2]
fM = (a0 / plx)**3 / q_bh[0]**2
M1 = 0.76                                          # mass of the giant [Msun] (Panuzzo et al. 2024)
M2 = brentq(lambda m: m**3 / (M1 + m)**2 - fM, 1, 200)
ours = dict(P=q_bh[0] * 365.25, e=q_bh[1], a0=a0, parallax=plx, i=inc, Omega=Om, omega=om, Tp=q_bh[2] * 365.25 + JD_REF)
# Panuzzo et al. (2024), Table 2, astrometry-only column: value and uncertainty
pub = dict(P=(4194.7, 112.3), e=(0.7262, 0.0056), a0=(27.07, 0.56), parallax=(1.6747, 0.0094), i=(110.659, 0.107),
           Omega=(136.200, 0.147), omega=(77.77, 0.66), Tp=(2458177.28, 0.98))
cmp = pd.DataFrame({k: dict(this_fit=ours[k], published=pub[k][0], sigma=pub[k][1],
                            pull=(ours[k] - pub[k][0]) / pub[k][1]) for k in pub}).T
print(cmp.round(4).to_string())
print(f'f_M = {fM:.2f} Msun (published 32.03 ± 0.64); with M1 = {M1} Msun: M2 = {M2:.1f} Msun (published, with RVs: 32.70 ± 0.82)')
# reproducibility: every element agrees with the published astrometric solution within 3 of its uncertainties
assert (cmp.pull.abs() < 3).all()

Our fit reproduces every published element within two of its uncertainties (column `pull`). The largest
differences are in the parallax (1.6σ) and therefore the mass function, which scales as $\varpi^{-3}$: we get
$f_M = 33.2\,M_\odot$, against $32.03 \pm 0.64$. The published solution also fitted the colour term and down-weighted
outliers, and it is the parallax that absorbs such differences. Either way the companion is a dark object of
about $33\,M_\odot$: a main-sequence star of that mass would outshine the giant by a factor of $10^4$.

Figure 5.3 shows the orbit. To display one-dimensional data on the sky we subtract the fitted single-star motion
(position, parallax, proper motion) from each observation; what is left is the orbit projected on the scan direction,
which we average per transit and compare with the model projected the same way.

In [ ]:
def orbit_al(s, p, q):
    """Along-scan orbital displacement of the model with parameters p (9) and (P, e, tp) = q at the observations s."""
    D, sn, cs, t = design5(s)
    X, Y = kepler_xy(t, *q)
    A, F, B, G = p[5:9]
    return (B * X + G * Y) * sn + (A * X + F * Y) * cs


D, sn, cs, t = design5(bh)
resid_orbit = bh.centroid_pos_al.to_numpy(float) - D @ p_bh[:5]          # data minus the single-star part
per = pd.DataFrame(dict(transit=bh.transit_id.to_numpy(), t=t, obs=resid_orbit, model=orbit_al(bh, p_bh, q_bh))) \
        .groupby('transit').mean()                                         # transit averages
tt = np.linspace(-3, 3, 2000)
X, Y = kepler_xy(tt, *q_bh)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(B * X + G * Y, A * X + F * Y, 'k-', lw=1)
x0, y0 = kepler_xy(np.array([per.t.min(), per.t.max()]), *q_bh)
ax[0].plot(B * x0 + G * y0, A * x0 + F * y0, 'o', color='C1', label='start and end of the DR4 data')
ax[0].plot(0, 0, 'k+', ms=12, label='centre of mass')
ax[0].set(xlabel='$\\Delta\\alpha_*$ [mas]', ylabel='$\\Delta\\delta$ [mas]', aspect='equal', title='fitted photocentre orbit')
ax[0].invert_xaxis(); ax[0].legend(fontsize=8)
ax[1].plot(2017.5 + per.t, per.obs, 'o', ms=4, label='data minus single-star motion (transit means)')
ax[1].plot(2017.5 + per.t, per.model, 'k.', ms=3, label='orbit model')
ax[1].set(xlabel='time [yr]', ylabel='along-scan orbital displacement [mas]'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Figure 5.3.** Left: the fitted photocentre orbit of Gaia BH3 on the sky (east to the left), $a_0 = 27$ mas, with the
start and end of the DR4 data (orange) and the centre of mass (cross). Right: along-scan positions minus the fitted
single-star motion, averaged per transit (blue), and the orbit model projected on each transit's scan direction (black).

The data and the model agree transit by transit, over a range of ±20 mas. The DR4 data cover the periastron passage
(early 2018) and 5.1 of the 11.5 years of the orbit (left); this is enough because a fast periastron passage fixes $e$ and
$T_p$, and the curvature of the remaining arc fixes $P$.

## 5.5 Why DR3 did not find it

Gaia DR3 used data from 25 July 2014 to 28 May 2017. BH3 had RUWE 3.4 there, but DR3 published no orbit and no
acceleration solution for it. We cut the DR4 data to the DR3 window and repeat the analysis.

In [ ]:
dr3 = bh[(bh.t > (2456863.5 - JD_REF) / 365.25) & (bh.t < (2457901.5 - JD_REF) / 365.25)]
u5 = np.sqrt(fit(dr3)[1] / fit(dr3)[2])
p7, c7, n7 = fit(dr3, accel=True)
prof_dr3, best3 = periodogram(dr3, P_grid)
print(f'DR3 window: {len(dr3)} observations, {dr3.transit_id.nunique()} transits; UWE 5-parameter {u5:.1f}, '
      f'with acceleration {np.sqrt(c7 / n7):.1f}; acceleration ({p7[5]:.2f}, {p7[6]:.2f}) mas/yr^2')
plt.semilogy(P_grid * 365.25, prof_dr4 - prof_dr4.min() + 1, 'o-', label='DR4 data (66 months)')
plt.semilogy(P_grid * 365.25, prof_dr3 - prof_dr3.min() + 1, 's-', label='the same, DR3 window only (34 months)')
plt.xscale('log'); plt.xlabel('period P [days]'); plt.ylabel('$\\chi^2_{\\min}(P) - \\chi^2_{\\min} + 1$'); plt.legend()
plt.show()
# DR4 pins the period (chi^2 rises by > 1e3 a factor of 2 away from the best period); the DR3 window does not (< 25)
far = np.abs(np.log(P_grid * 365.25 / 4200)) > np.log(2)
assert (prof_dr4[far] - prof_dr4.min()).min() > 1e3 and (prof_dr3 - prof_dr3.min()).max() < 25

**Figure 5.4.** Periodogram of Gaia BH3: the minimum of $\chi^2$ over eccentricity and periastron time at each trial
period, relative to the overall minimum (plus 1, for the logarithmic axis), for the DR4 data (blue) and for the same
data restricted to the DR3 time window (orange).

With DR4 data the minimum near 4200 d is sharp; $\chi^2$ is higher by $10^3$–$10^5$ at periods more than a factor of 2
away, and the deepest secondary minimum, near 2900 d, is worse by $\Delta\chi^2 \approx 200$: the period is pinned.
(The coarse grid of 25 periods is enough here; the refined fit of Section 5.3 does not depend on it.) In the
DR3 window, every period from 1500 to 30000 d fits equally well ($\Delta\chi^2 < 15$). Those 34 months fall between
two periastron passages, where the photocentre drifts slowly along a gently curved arc; a proper motion plus an
acceleration (UWE 2.4, against 8.5 without it) describes the arc almost as well as any orbit. DR3 could at best have
reported an acceleration; it reported nothing for this source. With 66 months and a periastron passage in the data,
the orbit — and a 33 $M_\odot$ black hole — follows.

This is the general lesson for DR4: the doubled baseline turns accelerations into orbits and RUWE-only candidates
into accelerations, exactly as Figure 1.4 predicted.

## Summary

- DR4 epoch astrometry gives, for each CCD crossing, the along-scan position, its uncertainty, the scan angle, the time
  and the parallax factor: everything needed to refit a star.
- The five-parameter model fits quasars and single stars to their uncertainties (UWE 0.98–1.04). Binaries do not.
- A Keplerian orbit is linear in the Thiele–Innes constants for fixed $(P, e, T_p)$: a grid in three parameters plus
  linear least squares fits it. For Gaia BH3 we reproduce the published astrometric orbit within 2σ, and
  $f_M \approx 33\,M_\odot$.
- In the 34 months of DR3 the same data constrain an acceleration but not the period.

**Exercises.** (1) Fit the orbit of HD 114762 (search periods from 20 to 1000 d). (2) Estimate the uncertainties of
the BH3 elements by refitting the orbit to bootstrap resamples of the transits (resample whole transits, not CCDs: why?).
(3) Gaia-4 has UWE 1.9: search its periodogram for the planet (periods 100–2000 d); compare $P$ and $a_0$ with
Stefánsson et al. (2025).